# Prepare simulated data for timepoint forecasting

In [1]:
import numpy as np
import pandas as pd
import os
import yaml

In [15]:
def get_analysis_start_date(analysis_date, n_days=180):
    """
    Get the start date for the analysis -- reach back in the past n_days.

    Parameters
    ----------
    analysis_date : pd.Timestamp
        The date of the analysis.
    n_days : int
        The number of days to reach back in the past.

    Returns
    -------
    pd.Timestamp
        The start date for the analysis.
    """
    start_date = analysis_date - pd.Timedelta(days=n_days)
    return start_date#.strftime('%Y-%m-%d')

In [16]:
def get_analysis_end_date(analysis_date, n_days=180):
    """ Obtain end date of an analysis period by removing n_days from the analysis_date (bias correction).

    Parameters
    ------------
    analysis_date : pd.Timestamp
        Analysis date.
    n_days : int
        Number of days to remove from the analysis_date.

    Returns
    -----------
    end_date : pd.Timestamp
        End date of the analysis period (i.e., last day of samples to include in the analysis folder).
    """
    end_date = analysis_date - pd.Timedelta(days=n_days)
    return end_date#.strftime('%Y-%m-%d')

In [17]:
ag_build = "flu-simulated-150k-samples-antigenic-clusters"
seq_build = "flu-simulated-150k-samples-sequence-clusters"

## Prep antigenic-based variants build

In [3]:
seqs_path = f"../data/{ag_build}/time-stamped/truth/seq_counts.tsv"
seqs = pd.read_csv(seqs_path, sep='\t')
seqs.head()

,date,country,variant,sequences
0,2019-12-08,tropics,2,1
1,2019-12-15,tropics,2,1
2,2019-12-22,north,2,2
3,2019-12-22,south,2,2
4,2019-12-22,tropics,2,6


In [4]:
cases_path = f"../data/{ag_build}/time-stamped/truth/case_counts.tsv"
cases = pd.read_csv(cases_path, sep='\t')
cases.head()

,date,country,cases
0,2020-01-05,north,225
1,2020-01-05,south,149
2,2020-01-05,tropics,576
3,2020-01-11,north,277
4,2020-01-11,south,119


In [5]:
min_date = seqs['date'].min()
max_date = seqs['date'].max()
print(min_date, max_date)

2019-12-08 2050-01-02


In [6]:
full_range = pd.date_range(start=min_date, end=max_date, freq='D')
timepoints = full_range[( (full_range.month == 4) | (full_range.month == 10) )& (full_range.day == 1) ]
timepoints

DatetimeIndex(['2020-04-01', '2020-10-01', '2021-04-01', '2021-10-01',
               '2022-04-01', '2022-10-01', '2023-04-01', '2023-10-01',
               '2024-04-01', '2024-10-01', '2025-04-01', '2025-10-01',
               '2026-04-01', '2026-10-01', '2027-04-01', '2027-10-01',
               '2028-04-01', '2028-10-01', '2029-04-01', '2029-10-01',
               '2030-04-01', '2030-10-01', '2031-04-01', '2031-10-01',
               '2032-04-01', '2032-10-01', '2033-04-01', '2033-10-01',
               '2034-04-01', '2034-10-01', '2035-04-01', '2035-10-01',
               '2036-04-01', '2036-10-01', '2037-04-01', '2037-10-01',
               '2038-04-01', '2038-10-01', '2039-04-01', '2039-10-01',
               '2040-04-01', '2040-10-01', '2041-04-01', '2041-10-01',
               '2042-04-01', '2042-10-01', '2043-04-01', '2043-10-01',
               '2044-04-01', '2044-10-01', '2045-04-01', '2045-10-01',
               '2046-04-01', '2046-10-01', '2047-04-01', '2047-10-01',
      

In [12]:
# Create a configuration yaml using the analysis/estimation dates, locations, and models
locations = seqs.country.unique().tolist()
estimation_dates = timepoints.strftime('%Y-%m-%d').tolist()
models = 

yaml_data = {
    'main': {
        'estimation_dates': estimation_dates,
        'locations': locations,
        'models': models
    }
}

with open("../config/config.yaml", 'w') as file:
    yaml.dump(yaml_data, file, default_flow_style=False)

In [14]:
seqs.date = pd.to_datetime(seqs.date)
cases.date = pd.to_datetime(cases.date)
for timepoint in timepoints:
    # Reach back 365 days for analysis and discard most recent 30 days for bias correction
    start_date = get_analysis_start_date(timepoint, n_days=365)
    end_date = get_analysis_start_date(timepoint, n_days=0) 
    analysis_date = timepoint#.strftime('%Y-%m-%d')

    # Subset dataframes to only include the interval
    timepoint_seqs = seqs[(seqs['date'] >= start_date) & (seqs['date'] <= end_date)]
    timepoint_cases = cases[(cases['date'] >= start_date) & (cases['date'] <= end_date)]

    # Write data if there are any sequences
    if len(timepoint_seqs) > 0:
        path = f'../data/{ag_build}/time-stamped/{analysis_date.strftime("%Y-%m-%d")}'
        os.makedirs(path, exist_ok=True)

        # If pathogen-embed used, replace -1 with 'other'
        timepoint_seqs.variant.replace(-1, 'other', inplace=True)

        # Write to file
        timepoint_seqs.to_csv(f'{path}/seq_counts.tsv', sep='\t', index=False)
        timepoint_cases.to_csv(f'{path}/case_counts.tsv', sep='\t',index=False)
    

/tmp/ipykernel_6320/317378337.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  timepoint_seqs.variant.replace(-1, 'other', inplace=True)


## Prep sequence-based variants build

In [18]:
seqs_path = f"../data/{seq_build}/time-stamped/truth/seq_counts.tsv"
seqs = pd.read_csv(seqs_path, sep='\t')
seqs.head()

,date,country,variant,sequences
0,2019-12-08,tropics,2,1
1,2019-12-15,tropics,15,1
2,2019-12-22,north,2,1
3,2019-12-22,north,15,1
4,2019-12-22,south,2,2


In [19]:
cases_path = f"../data/{seq_build}/time-stamped/truth/case_counts.tsv"
cases = pd.read_csv(cases_path, sep='\t')
cases.head()

,date,country,cases
0,2020-01-05,north,225
1,2020-01-05,south,149
2,2020-01-05,tropics,576
3,2020-01-11,north,277
4,2020-01-11,south,119


In [20]:
seqs.date = pd.to_datetime(seqs.date)
cases.date = pd.to_datetime(cases.date)
for timepoint in timepoints:
    # Reach back 365 days for analysis and discard most recent 30 days for bias correction
    start_date = get_analysis_start_date(timepoint, n_days=365)
    end_date = get_analysis_start_date(timepoint, n_days=0) 
    analysis_date = timepoint#.strftime('%Y-%m-%d')

    # Subset dataframes to only include the interval
    timepoint_seqs = seqs[(seqs['date'] >= start_date) & (seqs['date'] <= end_date)]
    timepoint_cases = cases[(cases['date'] >= start_date) & (cases['date'] <= end_date)]

    # Write data if there are any sequences
    if len(timepoint_seqs) > 0:
        path = f'../data/{seq_build}/time-stamped/{analysis_date.strftime("%Y-%m-%d")}'
        os.makedirs(path, exist_ok=True)

        # If pathogen-embed used, replace -1 with 'other'
        timepoint_seqs.variant.replace(-1, 'other', inplace=True)

        # Write to file
        timepoint_seqs.to_csv(f'{path}/seq_counts.tsv', sep='\t', index=False)
        timepoint_cases.to_csv(f'{path}/case_counts.tsv', sep='\t',index=False)

/tmp/ipykernel_6320/471478711.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  timepoint_seqs.variant.replace(-1, 'other', inplace=True)
